-----------

## **1. Setup and Imports**

In this notebook, we will learn how to interact with a locally running Large Language Model (LLM) using Python and Ollama.

Ollama allows us to run language models locally and interact with them through an API. We will use the `ollama` Python library to send prompts to our model and retrieve its responses.

For this project, we will use **Qwen 2.5 3B** as our local language model.

### **Objectives**

* Import the required Python library.
* Establish communication between Python and Ollama.
* Send prompts to the language model and inspect its responses.
* Build reusable functions for interacting with the model.

These fundamentals will prepare us for building a Retrieval-Augmented Generation (RAG) application later.


In [15]:
import ollama

-------------------------

## **2. Connecting to Ollama**

Before sending prompts, we need to ensure that Ollama is accessible from our Python environment.

Ollama runs as a local service and exposes an API that applications can use to interact with installed models.

In our setup, Ollama is running locally, and the `qwen2.5:3b` model is already installed.

### **Objective**

Verify that the Python application can communicate with Ollama and identify the available models.


In [23]:
# Check available models
ollama.list()

ListResponse(models=[Model(model='qwen2.5:3b', modified_at=datetime.datetime(2026, 10, 4, 13, 38, 51, 42809, tzinfo=TzInfo(10800)), digest='357c53fb659c5076de1d65ccb0b397446227b71a42be9d1603d46168015c9e4b', size=1929912432, details=ModelDetails(parent_model='', format='gguf', family='qwen2', families=['qwen2'], parameter_size='3.1B', quantization_level='Q4_K_M')), Model(model='llama3.2:3b', modified_at=datetime.datetime(2026, 7, 7, 22, 28, 7, 221331, tzinfo=TzInfo(10800)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M'))])

-----------------------

## **3. Sending Our First Prompt**

Now that Ollama is available, we can send our first prompt to the language model.

We will use `ollama.chat()` to provide a model name and a list of messages. Each message contains a role and its content.

In this example:

* `model` specifies which language model to use.
* `messages` contains the conversation input.
* `role="user"` identifies the message as a user request.
* `content` contains the actual prompt.

The model processes the prompt and returns a response object.

### **Objective**

Understand how to send a request to a local LLM and receive its response through Python.


In [ ]:
response = ollama.chat(
	model="qwen2.5:3b",
	messages=[
		{
			"role": "user",
			"content": "What is a Transformer in deep learning?"
		}
	]
)

response

-----------------

## **4. Understanding the Model Response**

The output of `ollama.chat()` is a response object, not just a plain text string.

This object contains information about the model, the generated message, and generation statistics.

To access the actual answer, we can use:

* `response.message` to access the generated message.
* `response.message.content` to extract the text of the answer.

This distinction is useful because applications usually need the generated text rather than the entire response object.

### **Objective**

Learn how to extract the model's answer from the response object.


In [ ]:
# Print only the response
response.message.content

'A Transformer is a type of deep learning model that is particularly well-suited for natural language processing (NLP) tasks. It was originally introduced by researchers from the Google Brain team in a 2017 paper. Unlike recurrent neural networks (RNNs), which can struggle with the sequential nature of text data, Transformers use a self-attention mechanism to process sequences of data, making them more flexible and efficient for NLP tasks such as language translation, text summarization, question answering, and more.\n\nHere are some key aspects of Transformers:\n\n1. **Self-Attention Mechanism**: This mechanism allows the model to weigh the importance of different elements within a sequence independently of the order in which they appear. In other words, it enables the model to focus on relevant parts of the input data at different times and positions. This is different from how RNNs process sequences, which always traverse through the sequence in a sequential manner, potentially miss

In [ ]:
# Get the message
response.message

Message(role='assistant', content='A Transformer is a type of deep learning model that is particularly well-suited for natural language processing (NLP) tasks. It was originally introduced by researchers from the Google Brain team in a 2017 paper. Unlike recurrent neural networks (RNNs), which can struggle with the sequential nature of text data, Transformers use a self-attention mechanism to process sequences of data, making them more flexible and efficient for NLP tasks such as language translation, text summarization, question answering, and more.\n\nHere are some key aspects of Transformers:\n\n1. **Self-Attention Mechanism**: This mechanism allows the model to weigh the importance of different elements within a sequence independently of the order in which they appear. In other words, it enables the model to focus on relevant parts of the input data at different times and positions. This is different from how RNNs process sequences, which always traverse through the sequence in a s

------------

## **5. Creating a Reusable Function**

Writing the same API call repeatedly makes code harder to maintain and reuse.

To solve this, we will create a function called `ask_llm()` that accepts a question, sends it to our local language model, and returns the generated answer.

The function will hide the implementation details of the API call behind a simple interface.

### **Example**

Instead of writing the full request every time, we will be able to call:

`ask_llm("What is an embedding?")`

### **Objective**

Encapsulate the Ollama interaction in a reusable function that can be extended as the project grows.


In [ ]:
def ask_llm(question):
	response = ollama.chat(
		model="qwen2.5:3b",
		messages=[
			{
				"role": "user",
				"content": question
			}
		]
	)

	return response.message.content

In [ ]:
ask_llm("What is an embedding?")

'An embedding is a type of mapping that allows data from a large, potentially high-dimensional space to be represented as a point in a lower-dimensional space, typically a much lower-dimensional space than the original one. The most common application is in natural language processing (NLP), where an embedding is a dense vector representation of words, phrases, or even entire documents.\n\nKey points about embeddings:\n\n1. Low Dimensionality: Embeddings represent high-dimensional data in a lower-dimensional space, often in a 3D, 2D, or even 1D space, depending on the specific application.\n\n2. Continuous and Dense: Embeddings are typically represented as a continuous vector in a high-dimensional space, allowing for the use of mathematical operations such as addition and subtraction.\n\n3. Learning-Based: Embeddings are often learned using machine learning techniques from large datasets, such as word embeddings in NLP, where the model learns to map words or phrases to vectors based on

In [ ]:
ask_llm("What is an embedding? Answer in one sentence.")

'An embedding is a technique used in machine learning to represent input data, such as text or images, in a high-dimensional vector space for easier processing and analysis.'

----------------

## **6. Controlling Text Generation**

The length of a generated response can affect both `readability` and `generation time`.

Ollama supports generation options that allow us to control aspects of the model's output.

In this section, we will use `num_predict` to set an upper limit on the number of tokens generated by the model.

We will extend `ask_llm()` with a `max_tokens` parameter so that we can choose a generation limit for each request.

### **Important Note**

`num_predict` limits the number of generated tokens. It does not guarantee a specific response length or generation speed, since performance also depends on the model and available hardware.

### **Objective**

Make our function more flexible by allowing callers to control the maximum number of generated tokens.


In [ ]:
def ask_llm(question, max_tokens=100):
	response = ollama.chat(
		model="qwen2.5:3b",
		messages=[
			{
				"role": "user",
				"content": question
			}
		],
		options={
			"num_predict": max_tokens
		}
	)

	return response.message.content

In [ ]:
ask_llm(
	"Explain self-attention in Transformers.",
	max_tokens=60
)

'Self-attention is a mechanism used in Transformers that allows the model to weigh the importance of different words in a sentence when computing a given word. It is a core component of the Transformer model, which was introduced in the paper "Attention Is All You Need" by Vaswani et al. ('

-------------

## **7. Building a Simple Chat Interface**

So far, we have been manually passing questions to our `ask_llm()` function.

In a real application, users should be able to provide their own questions without modifying the Python code.

In this section, we will build a simple command-line chat interface that repeatedly:

1. Accepts a question from the user.
2. Sends the question to our local LLM.
3. Displays the generated response.
4. Continues until the user chooses to exit.

This will give us a simple interactive interface for our local language model.

### **Objective**

Connect user input to our `ask_llm()` function and create a basic interactive chat experience.


In [ ]:
while True:

	question = input("You: ")

	if question.lower() in ["exit", "quit"]:
		print("Goodbye!")
		break
	
	answer = ask_llm(question, max_tokens = 100)

	print(f"Assistant: {answer}")


Assistant: A transformer is a widely used device in electrical engineering that changes the voltage of alternating current (AC) power. It can increase or decrease the voltage of an electrical circuit. This is achieved through the use of electromagnetic induction. Transformers are essential in power distribution systems because they allow for the efficient and safe transmission of electrical power over long distances. They are also used in electronic devices to step up or step down the voltage needed for specific components to operate correctly.

The term "transformer" is also used in
Goodbye!


----------------------

## **8. Conversation History**

Our current chat interface sends each user question as an independent request.

This means the model does not automatically receive the previous messages in the conversation.

For example, if the user asks:

* "What is a Transformer?"
* "What are its main components?"

The second question may be ambiguous because the model does not know that "its" refers to the Transformer mentioned in the previous message.

To maintain context, we need to store the conversation history and send the previous messages along with each new request.

We will represent the conversation as a list of messages containing different roles:

* `system` — instructions for the assistant.
* `user` — messages written by the user.
* `assistant` — responses generated by the model.

### **Objective**

Understand how conversation history is represented and use it to maintain context across multiple messages.


In [ ]:
def chat_with_history():
	messages = []

	while True :
		question = input("You: ")

		if question.lower() in ["exit", "quit"]:
			print("Goodbye!")
			break

		messages.append({
			"role": "user",
			"content": question
		})

		response = ollama.chat(
			model = "qwen2.5:3b",
			messages = messages,
			options={
				"num_predict": 100
			}
		)

		answer = response.message.content

		messages.append({
			"role": "assistant",
			"content": answer
		})

		print(f"Assistant: {answer}")

In [ ]:
chat_with_history()

Assistant: Okay, I'll be here if you need anything. Have a great day!
Goodbye!


--------------------------

## **9. Summary and Next Steps**

In this notebook, we learned how to interact with a locally running Large Language Model using Python and Ollama.

We started by connecting Python to Ollama and sending prompts to the `qwen2.5:3b` model. We then learned how to extract the generated response and built reusable functions for interacting with the model.

We also explored how to control text generation using `num_predict` and built a simple interactive chat interface.

Finally, we introduced conversation history by storing previous `user` and `assistant` messages and sending them back to the model with each new request. This allows the model to maintain context throughout a conversation.

### **What We Learned**

* How to communicate with Ollama from Python.
* How to send prompts using `ollama.chat()`.
* How to extract generated text from the response object.
* How to create reusable LLM functions.
* How to control the maximum number of generated tokens.
* How to build a simple command-line chat interface.
* How conversation history provides context across multiple messages.

### **Next Steps**

The next stage of this project is to move from a basic LLM application to a **Retrieval-Augmented Generation (RAG)** system.

Instead of asking the model questions without external knowledge, we will provide it with relevant information retrieved from our own documents.

------------------------

## **RAG Pipeline Architecture**

The following diagram illustrates the main stages of our Retrieval-Augmented Generation (RAG) system.

```text
Documents
	↓
Chunking
	↓
Embeddings
	↓
Vector Database
	↓
Retrieve Relevant Information
	↓
Context + User Question
	↓
Ollama / Qwen 2.5 3B
	↓
Answer
```

### How It Works

1. **Documents:** Load our knowledge sources, such as text files or PDFs.
2. **Chunking:** Split documents into smaller, manageable text segments.
3. **Embeddings:** Convert each text chunk into a numerical vector representing its semantic meaning.
4. **Vector Database:** Store the embeddings and their corresponding text chunks for efficient similarity search.
5. **Retrieve Relevant Information:** Find the chunks most relevant to the user's question.
6. **Context + User Question:** Combine the retrieved information with the user's original question.
7. **Ollama / Qwen 2.5 3B:** Send the question and retrieved context to our locally running language model.
8. **Answer:** Generate a response grounded in the retrieved information.


----------------------------

## **10. Preparing Our Documents**

Before building our RAG pipeline, we need to organize the documents that our application will use as a knowledge source.

In this step, we created a dedicated directory for storing our documents and added a sample text file called `student_handbook.txt`.

### **Project Structure**

```text
ollama-rag-project/
├── data/
│   └── documents/
│       └── student_handbook.txt
├── notebooks/
├── main.py
└── .gitignore
```

### **What We Did**

* Created a `data/` directory to store project data.
* Created a `documents/` subdirectory to organize our source documents.
* Created `student_handbook.txt` as a sample knowledge source.
* Added sample text about course registration, examinations, and graduation requirements.

### **Why Is This Important?**

A RAG application needs access to external information to answer questions using a custom knowledge source.

Organizing our documents separately from our Python code makes the project easier to maintain and extend as we add more documents.

### **Next Step: Document Loading**

Now that our sample document is ready, we will use Python to read its contents into a string.

This will allow us to process the text and prepare it for the next stage: **Chunking**.

```text
student_handbook.txt
		↓
	Python
		↓
	Text String
		↓
	 Chunking
```


In [1]:
# Check your path 
import os

print(os.getcwd())

c:\Users\User\ollama-rag-project\notebooks


In [ ]:
with open(
	"../data/documents/student_handbook.txt",
	"r",
	encoding="utf-8"
) as file:
	text = file.read()

print(text)

Students must register for their courses before the beginning of each semester.

Students are required to attend their scheduled examinations.

Students must complete all graduation requirements before receiving their degree.



----------------

## **11. Text Chunking**

After loading our document into Python, the next step in our RAG pipeline is **`text chunking`**.

Chunking is the process of splitting a large document into smaller pieces of text called **`chunks`**.

### **Why Do We Need Chunking?**

Real-world documents can contain a large amount of information. Sending an entire document to the language model for every question would be inefficient and could include a lot of irrelevant information.

Instead, we want to divide the document into smaller sections so that we can later retrieve only the parts that are relevant to a user's question.

For example:

```text
Document
│
├── Course Registration
├── Examination Rules
├── Attendance Requirements
├── Graduation Requirements
└── Student Services
```

If the user asks:

> "What are the graduation requirements?"

we want the retrieval system to find the section related to **Graduation Requirements** rather than processing the entire document.

### **Our Goal**

The goal of this step is to transform:

```text
Large Document
	  ↓
Smaller Chunks
```

These chunks will become the basic units that we later convert into **embeddings** and store in a **vector database**.

### **RAG Pipeline Progress**

At this point, our pipeline looks like:

```text
Documents
	↓
Document Loading
	↓
Chunking          ← We are here
	↓
Embeddings
	↓
Vector Database
	↓
Retrieve Relevant Information
	↓
Context + User Question
	↓
Ollama / Qwen 2.5 3B
	↓
Answer
```

In the next step, we will implement a simple chunking method and examine how the original document is divided into smaller pieces.


In [3]:
len(text)

227

In [ ]:
chunk_size = 100

chunks = []

for i in range(0, len(text), chunk_size):
	#print(f"i:{i}, i + chunk size: {i + chunk_size}")
	chunk = text[i:i + chunk_size]
	chunks.append(chunk)

chunks

['\ufeffStudents must register for their courses before the beginning of each semester.\n\nStudents are requi',
 'red to attend their scheduled examinations.\n\nStudents must complete all graduation requirements befo',
 're receiving their degree.\n']

As you can see the output is not that much clear so let's make it more `readable`.

In [ ]:
for i, chunk in enumerate(chunks):
	print(f"Chunk {i + 1}:")
	print(chunk)
	print("-" * 50)

Chunk 1:
﻿Students must register for their courses before the beginning of each semester.

Students are requi
--------------------------------------------------
Chunk 2:
red to attend their scheduled examinations.

Students must complete all graduation requirements befo
--------------------------------------------------
Chunk 3:
re receiving their degree.

--------------------------------------------------


### **Chunk Overlap**

Simple character-based chunking can sometimes split words, sentences, or ideas across different chunks.

For example:

```text
Chunk 1:
Students must complete all graduation require

Chunk 2:
ments before receiving their degree.
```

This can make individual chunks less meaningful because important information may be divided between them.

To reduce this problem, we can use **chunk overlap**.

Chunk overlap means that a small portion of one chunk is repeated in the next chunk.

For example:

```text
Chunk 1:
Students must complete all graduation requirements
before receiving their degree.

Chunk 2:
before receiving their degree.
Students must also...
```

The repeated text provides additional context between neighboring chunks.

### **Why Is Overlap Useful?**

Overlap helps preserve context when information spans across chunk boundaries.

This becomes particularly useful during retrieval because the retrieved chunk is more likely to contain enough surrounding context to represent the relevant information.

### **Concept**

```text
Chunk 1
┌──────────────────────────────┐
│        Main Content          │
└──────────────┬───────────────┘
               │
               │ Overlap
               ▼
┌──────────────────────────────┐
│ Overlapping │  New Content   │
└──────────────────────────────┘
Chunk 2
```



In the next step, we will implement chunk overlap and compare it with our previous chunking approach.


In [11]:
len(text)

227

In [ ]:
def split_text(text, chunk_size = 100, chunk_overlap = 20):

    chunks = []

    start = 0
 
    while start < len(text):  # 0 < 277 --> True || 80 < 277 --> True || 240 < 277 --> True || 320 < 277 --> False --> exit 
 
        end = start + chunk_size  # end = 0 + 100 --> end = 100 || end = 80 + 100 = 180   || end = 160 + 100 = 260   || end = 240 + 100 = 340
        chunk = text[start : end] # chunk = text[0:100]         || chunk = text[80 : 180] || chunk = text[160 : 260] || chunk = text[240 : 340]
        chunks.append(chunk)

        start = end - chunk_overlap  # start = 100 - 20 = 80    || start = 180 - 20 = 160 || start = 260 - 20 = 240  || start = 340 - 20 = 320

    return chunks 

In [12]:
chunks = split_text(
    text,
    chunk_size=100,
    chunk_overlap=20
)

In [13]:
for i, chunk in enumerate(chunks):
    print(f"chunk {i+1} :")
    print(chunk)
    print('-'*50)


chunk 1 :
﻿Students must register for their courses before the beginning of each semester.

Students are requi
--------------------------------------------------
chunk 2 :


Students are required to attend their scheduled examinations.

Students must complete all graduati
--------------------------------------------------
chunk 3 :
omplete all graduation requirements before receiving their degree.

--------------------------------------------------


-------------------------

## **12. Text Embeddings**

After splitting our document into smaller chunks, the next step in our RAG pipeline is to convert these text chunks into **embeddings**.

An embedding is a numerical representation of text that captures information about its semantic meaning.

For example:

```text
"What are the graduation requirements?"
                ↓
        Embedding Model
                ↓
     [0.12, -0.45, 0.88, ...]
```

The resulting vector contains numerical values that represent the text in a high-dimensional vector space.

### **Why Do We Need Embeddings?**

RAG systems need a way to determine which document chunks are most relevant to a user's question.

We can convert both the document chunks and the user's question into embeddings and then compare their vectors.

For example:

```text
Document Chunks
      ↓
Embedding Model
      ↓
Chunk Vectors

User Question
      ↓
Embedding Model
      ↓
Question Vector

Question Vector
      ↓
Similarity Comparison
      ↓
Most Relevant Chunks
```

If two pieces of text have similar meanings, their embedding vectors are generally closer together in the vector space.

This allows us to perform **semantic search**, where we search for meaning rather than relying only on exact keyword matches.

### **Embedding vs. Language Model**

Our project uses two different types of models for different purposes:

* **Embedding Model:** Converts text into numerical vectors for similarity search.
* **Qwen 2.5 3B:** Generates the final answer using the retrieved information.

The overall process will eventually look like:

```text
Documents
    ↓
Chunking
    ↓
Embeddings
    ↓
Vector Database
    ↓
Retrieve Relevant Information
    ↓
Context + User Question
    ↓
Qwen 2.5 3B
    ↓
Answer
```

### **Objective**

In this section, we will generate embeddings for our document chunks and inspect the resulting vectors.

These embeddings will later be used to build our semantic retrieval system.


### **Installing the Embedding Model**

For this project, we use **`nomic-embed-text`** as our local embedding model.

Unlike `qwen2.5:3b`, which is used to generate responses, `nomic-embed-text` is responsible for converting text into numerical vectors that can be used for semantic search.

The model must be downloaded locally through Ollama before generating embeddings.

If you have not downloaded the model yet, run the following command in your terminal:

```powershell
ollama pull nomic-embed-text
```

After the download is complete, you can verify that the model is available by running:

```powershell
ollama list
```

You should see `nomic-embed-text` in the list of installed models.

### **Why Do We Need This Model?**

Our RAG pipeline uses two different models for two different tasks:

* **`nomic-embed-text`** → converts text into embeddings for semantic search.
* **`qwen2.5:3b`** → generates the final answer using the retrieved context.

This separation allows the retrieval and generation stages of the RAG pipeline to use models designed for their specific tasks.


In [17]:
embedding_response  = ollama.embed(
    model= "nomic-embed-text",
    input= chunks[0]
)

In [18]:
embedding_response

EmbedResponse(model='nomic-embed-text', created_at=None, done=None, done_reason=None, total_duration=50280300, load_duration=4725200, prompt_eval_count=19, prompt_eval_duration=None, eval_count=None, eval_duration=None, embeddings=[[-0.0317717, 0.008476011, -0.18149692, 0.028415533, 0.04869547, 0.052696344, 0.03629947, 0.025471352, 0.022550317, -0.024567705, -0.0146597, 0.039017476, 0.06753937, 0.0178144, -0.026160197, -0.036287356, 0.004648722, -0.02413485, -0.0836744, -0.015785186, -0.0393833, -0.04764778, -0.032317095, -0.009213827, 0.058126245, 0.056566134, -0.026166055, -0.047708534, -0.013863461, 0.032325145, -0.00075172354, -0.023700666, 0.059595525, 0.00079665263, 0.001738852, -0.07784064, -0.007434742, -0.00086662144, -0.046672445, -0.046900444, 0.08710543, -0.01864541, 0.034378327, -0.020260006, -0.013054639, 0.027306, 0.061361983, -0.008444095, 0.06040033, 0.0017531557, -0.017250743, -0.056325443, -0.055979434, -0.010274218, 0.011993739, 0.037167724, -0.0061187707, 0.0317644

In [ ]:
# Take only embeddings
embedding_response.embeddings

[[-0.0317717,
  0.008476011,
  -0.18149692,
  0.028415533,
  0.04869547,
  0.052696344,
  0.03629947,
  0.025471352,
  0.022550317,
  -0.024567705,
  -0.0146597,
  0.039017476,
  0.06753937,
  0.0178144,
  -0.026160197,
  -0.036287356,
  0.004648722,
  -0.02413485,
  -0.0836744,
  -0.015785186,
  -0.0393833,
  -0.04764778,
  -0.032317095,
  -0.009213827,
  0.058126245,
  0.056566134,
  -0.026166055,
  -0.047708534,
  -0.013863461,
  0.032325145,
  -0.00075172354,
  -0.023700666,
  0.059595525,
  0.00079665263,
  0.001738852,
  -0.07784064,
  -0.007434742,
  -0.00086662144,
  -0.046672445,
  -0.046900444,
  0.08710543,
  -0.01864541,
  0.034378327,
  -0.020260006,
  -0.013054639,
  0.027306,
  0.061361983,
  -0.008444095,
  0.06040033,
  0.0017531557,
  -0.017250743,
  -0.056325443,
  -0.055979434,
  -0.010274218,
  0.011993739,
  0.037167724,
  -0.0061187707,
  0.031764474,
  -0.00615283,
  -0.038927466,
  0.1316095,
  0.07533722,
  -0.06633344,
  0.08906877,
  0.044530984,
  -0.019112

In [ ]:
# Take only the Victor 
embedding = embedding_response.embeddings[0]

embedding

[-0.0317717,
 0.008476011,
 -0.18149692,
 0.028415533,
 0.04869547,
 0.052696344,
 0.03629947,
 0.025471352,
 0.022550317,
 -0.024567705,
 -0.0146597,
 0.039017476,
 0.06753937,
 0.0178144,
 -0.026160197,
 -0.036287356,
 0.004648722,
 -0.02413485,
 -0.0836744,
 -0.015785186,
 -0.0393833,
 -0.04764778,
 -0.032317095,
 -0.009213827,
 0.058126245,
 0.056566134,
 -0.026166055,
 -0.047708534,
 -0.013863461,
 0.032325145,
 -0.00075172354,
 -0.023700666,
 0.059595525,
 0.00079665263,
 0.001738852,
 -0.07784064,
 -0.007434742,
 -0.00086662144,
 -0.046672445,
 -0.046900444,
 0.08710543,
 -0.01864541,
 0.034378327,
 -0.020260006,
 -0.013054639,
 0.027306,
 0.061361983,
 -0.008444095,
 0.06040033,
 0.0017531557,
 -0.017250743,
 -0.056325443,
 -0.055979434,
 -0.010274218,
 0.011993739,
 0.037167724,
 -0.0061187707,
 0.031764474,
 -0.00615283,
 -0.038927466,
 0.1316095,
 0.07533722,
 -0.06633344,
 0.08906877,
 0.044530984,
 -0.019112345,
 -0.022323538,
 0.02970403,
 0.03425021,
 0.0058148536,
 0.03

In [21]:
len(embedding)

768

In [22]:
# Print only the first 10 
embedding[:10]

[-0.0317717,
 0.008476011,
 -0.18149692,
 0.028415533,
 0.04869547,
 0.052696344,
 0.03629947,
 0.025471352,
 0.022550317,
 -0.024567705]

In [ ]:
# Put them together 
embedding = embedding_response.embeddings[0]

print("Embedding dimensions:", len(embedding))
print("First 10 values:", embedding[:10])

Embedding dimensions: 768
First 10 values: [-0.0317717, 0.008476011, -0.18149692, 0.028415533, 0.04869547, 0.052696344, 0.03629947, 0.025471352, 0.022550317, -0.024567705]


In [25]:
def embed_chunks(chunks):

    embeddings = []

    for chunk in chunks:
        response  = ollama.embed(
            model= "nomic-embed-text",
            input= chunk
        )
        embedding = response.embeddings[0]
        embeddings.append(embedding)
        
    return embeddings

In [26]:
embeddings = embed_chunks(chunks)

In [27]:
print("Number of chunks:", len(chunks))
print("Number of embeddings:", len(embeddings))
print("Dimensions of first embedding:", len(embeddings[0]))

Number of chunks: 3
Number of embeddings: 3
Dimensions of first embedding: 768


**Note:** This function converts each text chunk into a 768-dimensional embedding vector using `nomic-embed-text`. We extract the vector using `response.embeddings[0]` and store it in a list, keeping the embeddings in the same order as their original chunks.
